# 02 — Weather & Database
Building a DuckDB database and pulling weather data for every Mets home game from the Open-Meteo API (free, no API key needed).

In [1]:
import pandas as pd
import requests
import time

df = pd.read_csv("../data/mets_games_raw.csv")
print(f"Loaded {len(df)} games")
df_2026 = pd.read_csv("../data/mets_2026_prediction.csv")
print(f"Loaded {len(df_2026)} 2026 games")
df.head()

Loaded 337 games
Loaded 83 2026 games


,date,season,opponent,home_score,away_score,attendance,status,game_pk
0,2022-04-15,2022,Arizona Diamondbacks,10.0,3.0,43820.0,Final,662485
1,2022-04-16,2022,Arizona Diamondbacks,2.0,3.0,37935.0,Final,662626
2,2022-04-17,2022,Arizona Diamondbacks,5.0,0.0,24515.0,Final,662625
3,2022-04-18,2022,San Francisco Giants,NaN,NaN,NaN,Postponed,662616
4,2022-04-19,2022,San Francisco Giants,5.0,4.0,NaN,Final,662350


## Pulling Weather for 2022-2025 Game Dates
Citi Field coordinates are latitude 40.7571, longitude -73.8458.

Note: New York weather varies significantly — cold April/May games, summer heat and humidity, and rain are all meaningful signals unlike San Diego. We pull hourly temperature and precipitation around game time (4pm-8pm local time).

In [2]:
def get_weather(date, lat=40.7571, lon=-73.8458):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date,
        "end_date": date,
        "hourly": "temperature_2m,precipitation",
        "temperature_unit": "fahrenheit",
        "timezone": "America/New_York"
    }
    for attempt in range(3):
        try:
            resp = requests.get(url, params=params, timeout=10).json()
            break
        except requests.exceptions.RequestException as e:
            if attempt == 2:
                print(f"  Failed on {date} after 3 attempts: {e}")
                resp = {}
            else:
                time.sleep(2)
    hourly = resp.get("hourly", {})

    # Get conditions around game time (4pm-8pm = hours 16-20)
    temps = [t for t in hourly.get("temperature_2m", [])[16:21] if t is not None]
    precip = [p for p in hourly.get("precipitation", [])[16:21] if p is not None]

    return {
        "date": date,
        "avg_temp_f": round(sum(temps) / len(temps), 1) if temps else None,
        "avg_precip_mm": round(sum(precip) / len(precip), 1) if precip else None
    }

dates = df["date"].unique()
print(f"Pulling weather for {len(dates)} unique game dates...")

weather_data = []
for i, d in enumerate(dates):
    weather_data.append(get_weather(d))
    if (i + 1) % 25 == 0 or (i + 1) == len(dates):
        print(f"  {i + 1}/{len(dates)} dates pulled")

df_weather = pd.DataFrame(weather_data)
print("Done!")
df_weather.head()

Pulling weather for 324 unique game dates...
  25/324 dates pulled
  50/324 dates pulled
  75/324 dates pulled
  100/324 dates pulled
  125/324 dates pulled
  150/324 dates pulled
  175/324 dates pulled
  200/324 dates pulled
  225/324 dates pulled
  250/324 dates pulled
  275/324 dates pulled
  300/324 dates pulled
  324/324 dates pulled
Done!


,date,avg_temp_f,avg_precip_mm
0,2022-04-15,56.5,0.0
1,2022-04-16,56.7,0.0
2,2022-04-17,48.1,0.0
3,2022-04-18,45.5,0.1
4,2022-04-19,49.6,0.0


In [3]:
df_weather.to_csv("../data/mets_weather.csv", index=False)
print(f"Saved weather for {len(df_weather)} dates")
print(df_weather.describe())

Saved weather for 324 dates
       avg_temp_f  avg_precip_mm
count  324.000000     324.000000
mean    70.534877       0.157716
std     10.760125       0.494711
min     39.200000       0.000000
25%     63.900000       0.000000
50%     72.300000       0.000000
75%     78.200000       0.000000
max    101.700000       3.900000


In [4]:
# Pull weather for 2026 games already played
dates_2026 = df_2026[df_2026['status'] == 'Final']['date'].unique()
print(f"Pulling weather for {len(dates_2026)} 2026 game dates...")

weather_2026 = []
for i, d in enumerate(dates_2026):
    weather_2026.append(get_weather(d))
    if (i + 1) % 25 == 0 or (i + 1) == len(dates_2026):
        print(f"  {i + 1}/{len(dates_2026)} dates pulled")

df_weather_2026 = pd.DataFrame(weather_2026)
df_weather_2026.to_csv("../data/mets_weather_2026.csv", index=False)
print("Done!")
df_weather_2026.describe()

Pulling weather for 47 2026 game dates...
  25/47 dates pulled
  47/47 dates pulled
Done!


,avg_temp_f,avg_precip_mm
count,47.000000,47.000000
mean,66.189362,0.080851
std,13.002061,0.227116
min,40.100000,0.000000
25%,56.400000,0.000000
50%,70.400000,0.000000
75%,77.200000,0.000000
max,86.200000,1.000000


## Build DuckDB Database
Load games and weather data into a DuckDB database. DuckDB uses standard SQL syntax and is optimized for analytical queries — similar to how BigQuery works in production.

In [5]:
import duckdb

con = duckdb.connect("../data/mets.duckdb")

con.execute("CREATE OR REPLACE TABLE games AS SELECT * FROM read_csv_auto('../data/mets_games_raw.csv')")
con.execute("CREATE OR REPLACE TABLE weather AS SELECT * FROM read_csv_auto('../data/mets_weather.csv')")

print("Tables created:")
print(con.execute("SHOW TABLES").fetchdf())
print("Games rows:", con.execute("SELECT COUNT(*) FROM games").fetchone()[0])
print("Weather rows:", con.execute("SELECT COUNT(*) FROM weather").fetchone()[0])

Tables created:
      name
0    games
1  weather
Games rows: 337
Weather rows: 324


In [6]:
df_combined = con.execute("""
    SELECT
        g.date,
        g.season,
        g.opponent,
        g.home_score,
        g.away_score,
        g.attendance,
        g.status,
        g.game_pk,
        w.avg_temp_f,
        w.avg_precip_mm
    FROM games g
    LEFT JOIN weather w ON g.date = w.date
    WHERE g.status = 'Final'
    ORDER BY g.date
""").fetchdf()

print(f"Combined dataset: {len(df_combined)} games")
print(df_combined.head())

Combined dataset: 320 games
        date  season              opponent  home_score  away_score  \
0 2022-04-15    2022  Arizona Diamondbacks        10.0         3.0   
1 2022-04-16    2022  Arizona Diamondbacks         2.0         3.0   
2 2022-04-17    2022  Arizona Diamondbacks         5.0         0.0   
3 2022-04-19    2022  San Francisco Giants         3.0         1.0   
4 2022-04-19    2022  San Francisco Giants         5.0         4.0   

   attendance status  game_pk  avg_temp_f  avg_precip_mm  
0       43820  Final   662485        56.5            0.0  
1       37935  Final   662626        56.7            0.0  
2       24515  Final   662625        48.1            0.0  
3       27490  Final   662616        49.6            0.0  
4        <NA>  Final   662350        49.6            0.0  


In [7]:
df_combined.to_csv("../data/mets_master.csv", index=False)
print("Saved mets_master.csv")

Saved mets_master.csv
